In [1]:
import sys
sys.path.append('..')

from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
import pandas as pd
import pickle
from scipy.sparse import save_npz

# Load your data
movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

print("Original data shapes:")
print(f"Movies: {movies.shape}, Ratings: {ratings.shape}")

# Initialize and run preprocessing
preprocessor = DataPreprocessor(movies, ratings)
movies_processed, ratings_processed, user_item_matrix = preprocessor.get_preprocessed_data()

# Display results
print("\n=== Preprocessed Data Summary ===")
print(f"Processed movies: {len(movies_processed)}")
print(f"Processed ratings: {len(ratings_processed)}")

# Feature Engineering (only if dataset is not too large)
if len(movies_processed) < 10000:  # Only for smaller datasets
    feature_engineer = FeatureEngineer(movies_processed)
    genre_features, genre_names = feature_engineer.create_genre_features()
    tfidf_features = feature_engineer.create_content_features()
else:
    print("Dataset too large for content features. Skipping feature engineering.")
    genre_features, tfidf_features = None, None

# Save processed data
print("\nSaving processed data...")
movies_processed.to_csv('../data/movies_processed.csv', index=False)
ratings_processed.to_csv('../data/ratings_processed.csv', index=False)

# Save sparse matrix efficiently
save_npz('../data/user_item_matrix_sparse.npz', user_item_matrix)

if genre_features is not None:
    save_npz('../data/genre_features_sparse.npz', genre_features)
if tfidf_features is not None:
    save_npz('../data/tfidf_features_sparse.npz', tfidf_features)

# Save encoders for later use
with open('../data/user_encoder.pkl', 'wb') as f:
    pickle.dump(preprocessor.user_encoder, f)
    
with open('../data/movie_encoder.pkl', 'wb') as f:
    pickle.dump(preprocessor.movie_encoder, f)

print("All processed data saved successfully!")

Original data shapes:
Movies: (86537, 3), Ratings: (33832162, 4)
Preprocessing movies data...
Found 20 unique genres: {'(no genres listed)', 'Adventure', 'Sci-Fi', 'Fantasy', 'Documentary', 'Animation', 'Drama', 'IMAX', 'War', 'Comedy', 'Romance', 'Action', 'Thriller', 'Crime', 'Film-Noir', 'Western', 'Horror', 'Mystery', 'Children', 'Musical'}
Preprocessing ratings data...
After filtering: 33624935 ratings
Creating sparse user-item matrix...
Sparse user-item matrix shape: (307411, 32004)
Sparsity: 99.66%
Number of ratings: 33624935
Matrix size: 9838381644

=== Preprocessed Data Summary ===
Processed movies: 86537
Processed ratings: 33624935
Dataset too large for content features. Skipping feature engineering.

Saving processed data...
All processed data saved successfully!
